In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import time
from datetime import datetime


import sys
import os

In [2]:
# MAGIA (não mexer mt pfv)
def show_args(func):
    def wrapper(*args, **kwargs):
        arg_list = []

        # Prepare argument list
        for name, value in zip(func.__code__.co_varnames, args):
            if isinstance(value, pd.DataFrame):
                arg_list.append(f"{name}: DataFrame ({value.shape})")
            else:
                arg_list.append(f"{name}: {value}")
        for k, v in kwargs.items():
            if isinstance(v, pd.DataFrame):
                arg_list.append(f"{k}: DataFrame ({v.shape})")
            else:
                arg_list.append(f"{k}: {v}")

        #print(f"Executando <{func.__name__}>\n({'\\n'.join(arg_list)})")

        # Measure execution time
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()

        print(f"Tempo de execução: {end - start:.6f} segundos\n")
        return result
    return wrapper

In [3]:
from pathlib import Path
import pandas as pd

@show_args
def csvs_to_parquet(input_folder, output_folder):
    input_path = Path(input_folder)
    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)

    for csv_file in input_path.glob("*.csv"):
        try:
            df = pd.read_csv(csv_file, encoding="utf-8")
        except UnicodeDecodeError:
            df = pd.read_csv(csv_file, encoding="latin1")

        parquet_file = output_path / csv_file.with_suffix(".parquet").name
        df.to_parquet(parquet_file, index=False)



In [4]:
#csvs_to_parquet("../data/data/refined/csvs", "../data/data/refined/parquets")

In [5]:
articles_raw = pd.read_parquet('../data/data/refined/parquets/article.parquet').rename(columns={
    'eid': 'art_id',
    'subtype description': 'art_subtype',
    'title': 'art_title',
    'published date': 'art_published_date',
    'cited by count': 'art_citations',
    'source id' : 'art_source',
    'aggregation type' : 'art_source_type'
})[['art_id', 'art_subtype', 'art_title', 'art_published_date', 'art_citations', 'art_source', 'art_source_type']] # B

scores_raw = pd.read_parquet('../data/data/refined/parquets/CiteScore_2024_annual_values.parquet').rename(columns={
    'Scopus Source ID' : 'art_source',
    'Title' : 'src_title',
    'Citation Count' : 'src_citation_count',
    'CiteScore' : 'src_citescore',
    'Scopus Sub-Subject Area' : 'src_subject_area',
    'Percentile' : 'src_percentile',
    'RANK' : 'src_rank',
    'Rank Out Of' : 'src_rank_count',
    'Quartile' : 'src_quartile'
})[['art_source', 'src_title', 'src_citation_count', 'src_citescore', 'src_subject_area', 'src_percentile', 'src_rank', 'src_rank_count', 'src_quartile']]


authors_institutions_raw = pd.read_parquet('../data/data/refined/parquets/authors_institution.parquet').rename(columns={
    #eid,auid,creator,afid,dptid,organization,country,city
    'eid': 'art_id',
    'auid': 'aut_id',
    'creator': 'main_writer',
    'afid': 'ins_id',
    'dptid': 'dpt_id',
    'organization': 'ins_org',
    'country': 'ins_country',
    'city': 'ins_city'
})

institutions_raw = (pd.read_parquet('../data/data/refined/parquets/institutions.parquet').rename(columns={
        'Código Mantenedora': 'ins_maint_id',
        # 'Razão Social': 'ins_legal_name',
        # 'Município': 'ins_city',
        # 'UF': 'ins_state',
        # 'Tipo de Credenciamento': 'ins_modalidade',
        'Categoria': 'ins_category',
        # 'CI': 'ins_ci',
        # 'Ano CI': 'ins_ci_year',
        # 'CI-EaD': 'ins_ci_ead',
        # 'Ano CI-EaD': 'ins_ci_ead_year',
        # 'IGC': 'ins_igc',
        'afid': 'ins_id',
        # 'Ano IGC': 'ins_igc_year',
        'Situação da IES': 'ins_status',
        'Código IES': 'ins_code',
        'name': 'ins_name'
    })
    [['ins_maint_id', 'ins_category', 'ins_id', 'ins_status', 'ins_code', 'ins_name']]
)

#articles_authors_raw = pd.read_parquet('../data/data/refined/parquets/articles_author.parquet').rename(columns={
#    'eid': 'art_id',
#    'auid': 'aut_id',
#    'creator': 'main_writer'
#})[['art_id', 'aut_id', 'main_writer']]
#
authors_raw = pd.read_parquet('../data/data/refined/parquets/author.parquet').rename(columns={
    'auid': 'aut_id',
    'given name': 'aut_name',
    'surname': 'aut_surname',
    'indexed name': 'aut_indexed_name'
})[['aut_id', 'aut_name', 'aut_surname', 'aut_indexed_name']]

# departments_raw = pd.read_parquet('../data/data/refined/parquets/departments.parquet').rename(columns={
#     'afid' : 'ins_id',
#     'dptid': 'dpt_id',
#     'organization': 'dpt_name',
#     'country' : 'dpt_country',
#     'city' : 'dpt_city'
# })[['ins_id', 'dpt_id', 'dpt_name', 'dpt_country', 'dpt_city']]

# articles_subject_areas_raw = pd.read_parquet('../data/data/refined/parquets/article_subject_areas.parquet').rename(columns={
#     'eid': 'art_id',
#     'subject area code': 'sub_id'
# })[['art_id', 'sub_id']] # A

subject_areas_raw = pd.read_parquet('../data/data/refined/parquets/subject_area.parquet')
subject_areas_raw.columns = ['sub_id', 'src_subject_area', 'src_subject_macro'] # A

# articles_institutions_raw = pd.read_parquet('../data/data/refined/parquets/articles_institution.parquet').rename(columns={
#     'eid': 'art_id',
#     'afid': 'ins_id',
#     'creator': 'main_writer',
#     'country': 'ins_country'
# })[['art_id', 'ins_id', 'main_writer', 'ins_country']]




In [6]:
main = (
articles_raw
    .merge(scores_raw, on='art_source', how='inner')
    .merge(authors_institutions_raw, on='art_id', how='inner')
    .merge(institutions_raw, on='ins_id', how='inner')
    .merge(authors_raw, on='aut_id', how='inner')
    .merge(subject_areas_raw, on='src_subject_area', how='inner')
    #.merge(authors_raw, on='aut_id', how='inner')
    #.merge(institutions_raw, on='art_id', how='inner')
    # .merge(subjects_info, on='art_id', how='inner')
    # .merge(articles_raw, on='art_source', how='inner')
)

#articles_institutions_raw['ins_id'] = articles_institutions_raw['ins_id'].astype(str)
#institutions_raw['ins_id'] = institutions_raw['ins_id'].astype(str)
#departments_raw['ins_id'] = departments_raw['ins_id'].astype(str)
#main = main.merge(departments_info, on='ins_id', how='left')


In [7]:
institutions_raw['ins_id'].value_counts()

ins_id
60003597     31
60022378     31
60281103     21
60019608      8
60021699      8
             ..
118511108     1
118626976     1
101929885     1
106310222     1
131888906     1
Name: count, Length: 1159, dtype: int64

In [8]:
main.head()

,art_id,art_subtype,art_title,art_published_date,art_citations,art_source,art_source_type,src_title,src_citation_count,src_citescore,...,ins_maint_id,ins_category,ins_status,ins_code,ins_name,aut_name,aut_surname,aut_indexed_name,sub_id,src_subject_macro
0,2-s2.0-84951935294,Article,Body adiposity index performance in estimating...,2015-12-30,18.0,12653.0,Journal,Nutrition Journal,2502,6.7,...,15715,Publica,Ativa,55,Universidade de São Paulo,Giliane,Belarmino,Belarmino G.,2701,MEDI
1,2-s2.0-84951935294,Article,Body adiposity index performance in estimating...,2015-12-30,18.0,12653.0,Journal,Nutrition Journal,2502,6.7,...,15715,Publica,Ativa,55,Universidade de São Paulo,Lilian Mika,Horie,Horie L.M.,2701,MEDI
2,2-s2.0-84951935294,Article,Body adiposity index performance in estimating...,2015-12-30,18.0,12653.0,Journal,Nutrition Journal,2502,6.7,...,15715,Publica,Ativa,55,Universidade de São Paulo,Priscila Campos,Sala,Sala P.C.,2701,MEDI
3,2-s2.0-84951935294,Article,Body adiposity index performance in estimating...,2015-12-30,18.0,12653.0,Journal,Nutrition Journal,2502,6.7,...,15715,Publica,Ativa,55,Universidade de São Paulo,Raquel S.,Torrinhas,Torrinhas R.S.,2701,MEDI
4,2-s2.0-84951935294,Article,Body adiposity index performance in estimating...,2015-12-30,18.0,12653.0,Journal,Nutrition Journal,2502,6.7,...,15715,Publica,Ativa,55,Universidade de São Paulo,Dan L.,Waitzberg,Waitzberg D.L.,2701,MEDI


In [9]:
teste = main.head(6000)
#teste.to_csv('teste.csv')

In [10]:
main.columns

Index(['art_id', 'art_subtype', 'art_title', 'art_published_date',
       'art_citations', 'art_source', 'art_source_type', 'src_title',
       'src_citation_count', 'src_citescore', 'src_subject_area',
       'src_percentile', 'src_rank', 'src_rank_count', 'src_quartile',
       'aut_id', 'main_writer', 'ins_id', 'dpt_id', 'ins_org', 'ins_country',
       'ins_city', 'ins_maint_id', 'ins_category', 'ins_status', 'ins_code',
       'ins_name', 'aut_name', 'aut_surname', 'aut_indexed_name', 'sub_id',
       'src_subject_macro'],
      dtype='object')

In [11]:
ranking = main[['art_id', 'art_subtype', 'art_published_date', 'src_subject_area', 'src_subject_macro', 'src_quartile', 'ins_maint_id', 'ins_name', 'ins_category', 'ins_country', 'aut_id']]
institution = main[['ins_maint_id', 'ins_name', 'ins_category', 'ins_status']].drop_duplicates(subset=['ins_maint_id'])


print(institution.shape)

(571, 4)


In [12]:
print(institution[(institution['ins_name'] == 'Universidade Federal Fluminense')])

    ins_maint_id                         ins_name ins_category ins_status
15         15589  Universidade Federal Fluminense      Publica      Ativa


In [13]:
institution.head()

,ins_maint_id,ins_name,ins_category,ins_status
0,15715,Universidade de São Paulo,Publica,Ativa
15,15589,Universidade Federal Fluminense,Publica,Ativa
19,14678,Universidade Federal de Santa Maria,Publica,Ativa
21,15515,Universidade Federal de Minas Gerais,Publica,Ativa
25,283,Universidade de Pernambuco,Publica,Ativa


In [14]:
ranking.head()

,art_id,art_subtype,art_published_date,src_subject_area,src_subject_macro,src_quartile,ins_maint_id,ins_name,ins_category,ins_country,aut_id
0,2-s2.0-84951935294,Article,2015-12-30,Medicine (miscellaneous),MEDI,1,15715,Universidade de São Paulo,Publica,Brazil,56862304200
1,2-s2.0-84951935294,Article,2015-12-30,Medicine (miscellaneous),MEDI,1,15715,Universidade de São Paulo,Publica,Brazil,16241649700
2,2-s2.0-84951935294,Article,2015-12-30,Medicine (miscellaneous),MEDI,1,15715,Universidade de São Paulo,Publica,Brazil,54400149800
3,2-s2.0-84951935294,Article,2015-12-30,Medicine (miscellaneous),MEDI,1,15715,Universidade de São Paulo,Publica,Brazil,6506627898
4,2-s2.0-84951935294,Article,2015-12-30,Medicine (miscellaneous),MEDI,1,15715,Universidade de São Paulo,Publica,Brazil,7006460075


In [15]:
ranking = ranking[(ranking['src_quartile'] == 4)]

In [16]:
ranking['src_quartile'].value_counts()

src_quartile
4    614250
Name: count, dtype: int64

In [17]:
ranking_article = ranking[(ranking['art_subtype'] == 'Article')]
#ranking_article['DI_value'] = (ranking_article['src_percentile'] / ranking_article.groupby('art_id')['aut_id'].transform('count'))
ranking_article['DI_value'] = (1/(ranking_article.groupby('art_id')['aut_id'].transform('count') * ranking_article['src_quartile']/100))

C:\Users\aline\AppData\Local\Temp\ipykernel_3324\2761021501.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ranking_article['DI_value'] = (1/(ranking_article.groupby('art_id')['aut_id'].transform('count') * ranking_article['src_quartile']/100))


In [18]:
#ranking.head(100000).to_csv('rankings.csv', index=False)

In [19]:
# --- subject-level ---
sub_sums = ranking_article.groupby(['ins_maint_id', 'src_subject_area'])['DI_value'].sum()
pivoted_area = sub_sums.unstack(fill_value=0).reset_index()

di_table_subs = (
    institution
    .merge(pivoted_area, on='ins_maint_id', how='left')
    .fillna(0)
)

# --- macro-level (completely separate) ---
macro_sums = ranking_article.groupby(['ins_maint_id', 'src_subject_macro'])['DI_value'].sum()
pivoted_macro = macro_sums.unstack(fill_value=0).reset_index()

di_table_macros = (
    institution
    .merge(pivoted_macro, on='ins_maint_id', how='left')
    .fillna(0)
)

In [20]:
print(sub_sums)

ins_maint_id  src_subject_area                                    
1             Accounting                                                31.666667
              Advanced and Specialized Nursing                          25.000000
              Aging                                                     12.500000
              Agricultural and Biological Sciences (miscellaneous)     345.694444
              Agronomy and Crop Science                               1069.494709
                                                                         ...     
18557         Public Health, Environmental and Occupational Health       3.571429
              Surgery                                                    2.083333
19572         Education                                                  6.250000
19664         Economics and Econometrics                                 6.250000
              Marketing                                                  6.250000
Name: DI_value, Length: 21805, 

In [21]:
di_table_subs.head()

,ins_maint_id,ins_name,ins_category,ins_status,Accounting,Acoustics and Ultrasonics,Advanced and Specialized Nursing,Aerospace Engineering,Aging,Agricultural and Biological Sciences (miscellaneous),...,Toxicology,Transplantation,Transportation,Urban Studies,Urology,Veterinary (miscellaneous),Virology,Visual Arts and Performing Arts,Waste Management and Disposal,Water Science and Technology
0,15715,Universidade de São Paulo,Publica,Ativa,1717.053699,0.000000,72.321429,0.000000,238.125000,141.274802,...,117.986111,0.000000,108.333333,642.073413,72.083333,475.833333,113.333333,669.750000,2099.783399,932.099206
1,15589,Universidade Federal Fluminense,Publica,Ativa,63.244048,0.000000,0.000000,0.000000,0.000000,17.916667,...,4.166667,8.333333,50.000000,283.645833,0.000000,50.000000,3.125000,118.750000,196.130952,5.000000
2,14678,Universidade Federal de Santa Maria,Publica,Ativa,335.625000,16.666667,0.000000,0.000000,0.000000,54.253472,...,45.352564,0.000000,0.000000,143.363095,0.000000,98.184524,0.000000,19.583333,327.986111,804.583333
3,15515,Universidade Federal de Minas Gerais,Publica,Ativa,460.937500,50.000000,21.428571,8.333333,69.871795,35.768849,...,112.142857,0.000000,18.750000,177.357955,19.166667,999.581530,6.250000,161.597222,1572.559524,239.583333
4,283,Universidade de Pernambuco,Publica,Ativa,0.000000,5.000000,0.000000,5.000000,17.708333,6.250000,...,0.000000,0.000000,0.000000,16.194444,0.000000,0.000000,0.000000,1.250000,43.750000,60.000000


In [22]:
di_table_subs = di_table_subs.drop(['ins_maint_id', 'ins_category', 'ins_status'], axis=1)
di_table_macros = di_table_macros.drop(['ins_maint_id', 'ins_category', 'ins_status'], axis=1)
di_table_macros.head()

,ins_name,AGRI,ARTS,BIOC,BUSI,CHEM,COMP,DECI,DENT,EART,...,MATH,MEDI,MULT,NEUR,NURS,PHAR,PHYS,PSYC,SOCI,VETE
0,Universidade de São Paulo,12255.195978,8133.555556,20818.747741,5440.599334,2355.406746,2489.400253,2004.723214,272.291667,4399.279070,...,5629.715909,32436.408412,1263.152056,4057.870320,2679.990049,437.193813,2085.947421,4338.313492,45897.140494,487.083333
1,Universidade Federal Fluminense,1331.516955,2856.443834,1463.083113,1093.263889,209.604978,601.753247,844.950397,25.000000,1746.314484,...,930.505952,4009.153234,89.583333,214.117063,181.488095,233.112859,271.011905,646.388889,12227.290449,50.000000
2,Universidade Federal de Santa Maria,7289.421769,1044.166667,2467.056277,652.638889,266.418651,133.333333,159.583333,0.000000,1471.438492,...,59.464286,1752.329771,163.392857,293.055556,264.957265,676.256452,149.340278,340.416667,6004.136905,98.184524
3,Universidade Federal de Minas Gerais,3670.555930,3017.500000,4920.212386,2085.592082,770.426333,1488.055556,394.821429,5.000000,1552.723214,...,1358.055556,8012.505521,263.541667,1141.235897,672.445136,406.512272,402.384259,1494.199735,19530.115028,1037.081530
4,Universidade de Pernambuco,223.869048,150.972222,608.457341,95.083333,0.000000,5.000000,57.916667,4.166667,548.065476,...,0.000000,1804.987073,25.000000,109.791667,243.898810,26.875000,30.000000,23.333333,1450.994138,0.000000


In [23]:
#print(institution['ins_maint_id'].value_counts())

In [24]:
# def gini(values):
#     values = np.array(values, dtype=float)
#     if np.amin(values) < 0:
#         values -= np.amin(values)
#     values += 1e-9  # avoid division by zero
#     values = np.sort(values)
#     n = values.size
#     return (2 * np.sum(np.arange(1, n+1) * values) / (n * np.sum(values))) - (n + 1) / n

# def row_gini(row):
#     return gini(row[1:])  # skip ins_id

# di_table_subs['gini_index'] = di_table_subs.apply(row_gini, axis=1)
# di_table_macros['gini_index'] = di_table_macros.apply(row_gini, axis=1)


In [25]:
# di_table_subs = di_table_subs[(di_table_subs['gini_index'] >= 0.05) & (di_table_subs['gini_index'] <= 0.95)]
# di_table_macros = di_table_macros[(di_table_macros['gini_index'] >= 0.05) & (di_table_macros['gini_index'] <= 0.95)]
# di_table_subs.sort_values('gini_index', inplace=True)
# di_table_macros.sort_values('gini_index', inplace=True)
# di_table_subs.drop_duplicates(subset='ins_name', inplace=True)
# di_table_macros.drop_duplicates(subset='ins_name', inplace=True)

In [26]:
# plt.figure(figsize=(18, 8))
# plt.plot(di_table_subs['ins_name'].head(25), di_table_subs['gini_index'].head(25), linestyle=(0, (1, 0)))
# plt.xticks(rotation=90, fontsize=10)  # rotate + smaller font
# plt.ylabel('Gini Index')
# plt.title('Gini Index per Institution')
# plt.grid(True)
# plt.show()

In [27]:
# plt.figure(figsize=(18, 8))
# plt.plot(di_table_macros['ins_name'].head(50), di_table_macros['gini_index'].head(50), linestyle=(0, (1, 0)))
# plt.xticks(rotation=90, fontsize=10)  # rotate + smaller font
# plt.ylabel('Gini Index')
# plt.title('Gini Index per Institution')
# plt.grid(True)
# plt.show()

In [28]:
#di_table_subs.head()

In [29]:
#di_table_macros.head(20)

In [30]:
def order_by_non_zero_count(df):
    """
    Orders DataFrame rows by the number of non-zero values in ascending order (fewest zeros first).
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe to be sorted
    
    Returns:
    --------
    pd.DataFrame
        DataFrame sorted by non-zero count (rows with more non-zero values come first)
    """
    non_zero_counts = (df != 0).sum(axis=1)
    # Sort by non-zero count in descending order (most non-zero values first)
    return df.iloc[non_zero_counts.argsort()[::-1]]


"""
#TEST CASE
test_df = pd.DataFrame({
    'col1': [1, 1, 0],
    'col2': [0, 1, 0],
    'col3': [0, 1, 1],
    'col4': [4430, 1, 0],
}, index=['A', 'B', 'C'])

print("Original DataFrame:")
print(test_df)
print("\nOrdered DataFrame:")
print(order_by_non_zero_count(test_df))"""

'\n#TEST CASE\ntest_df = pd.DataFrame({\n    \'col1\': [1, 1, 0],\n    \'col2\': [0, 1, 0],\n    \'col3\': [0, 1, 1],\n    \'col4\': [4430, 1, 0],\n}, index=[\'A\', \'B\', \'C\'])\n\nprint("Original DataFrame:")\nprint(test_df)\nprint("\nOrdered DataFrame:")\nprint(order_by_non_zero_count(test_df))'

In [31]:
final_leys_macros = order_by_non_zero_count(di_table_macros)[:249]
final_leys_macros['ins_name'].to_csv('./data/macro/institutions_names.csv', index=False)
final_leys_macros = final_leys_macros.drop(columns=['ins_name'])
final_leys_macros.columns.to_frame(index=False).to_csv('./data/macro/labels.csv', index=False, header=False)
final_leys_macros = final_leys_macros.T.to_csv('./data/macro/rank.txt', index=False, header=False)

final_leys_subs = order_by_non_zero_count(di_table_subs)[:249]
final_leys_subs['ins_name'].to_csv('./data/sub/institutions_names.csv', index=False)
final_leys_subs = final_leys_subs.drop(columns=['ins_name'])
final_leys_subs.columns.to_frame(index=False).to_csv('./data/sub/labels.csv', index=False, header=False)
final_leys_subs = final_leys_subs.T.to_csv('./data/sub/rank.txt', index=False, header=False)